> ## MODEL LAMA (v1, 11 kelas) -- notebook backup, TERISOLASI dari training baru
>
> Checkpoint di-load dari `checkpoints/v1_11class_backup/last.pth` (bukan `checkpoints/best.pth` yang sekarang berisi model BARU), dan kalau dijalankan ulang, index katalog hasil regenerasi ditulis ke `data/cache/v1_11class_backup/catalog_embeddings_regenerated.npz` -- **TIDAK** menimpa `data/cache/catalog_embeddings.npz` (live) maupun katalog backup asli (`catalog_embeddings.npz` di folder yang sama).
>
> **Catatan:** `source_csv` (`data/raw/metadata.csv`) sekarang sudah bertambah jadi 33.937 baris (dulu 5.659 saat notebook ini pertama dijalankan) -- kalau di-run ulang, akan meng-embed SEMUA baris itu pakai model lama, bukan cuma 5.659 yang asli. Biasanya **tidak perlu dijalankan ulang** karena hasil aslinya sudah ada di `data/cache/v1_11class_backup/catalog_embeddings.npz`.
>
> Untuk retrieval model TERBARU, pakai `notebooks/02_embedding_retrieval.ipynb` yang asli, bukan file di folder ini.

# 02 — Embedding & Evaluasi Retrieval (Visual Search)

Klasifikasi style (`01_train.ipynb`) itu **pretext task**. Metrik sukses
SEBENARNYA untuk Visual Search adalah: kalau user upload foto karya, apakah
sistem menemukan karya yang **mirip** dari katalog?

Notebook ini:
1. Ekstrak embedding (`forward_features`, sebelum classifier head) dari
   checkpoint terbaik.
2. **Evaluasi retrieval**: gallery = train+val (katalog), query = test
   (foto "baru"). Ukur **Recall@k / Precision@k / mAP@k** — pakai label style
   sebagai proksi "relevan" (karya se-style dianggap mirip).
3. Bandingkan ke **baseline acak** (chance) supaya jelas seberapa jauh di atas tebakan.
4. Tampilkan contoh: 1 query -> top-5 hasil retrieval (yang sebenarnya dilihat user).
5. Bangun index embedding **seluruh katalog** (`data/cache/catalog_embeddings.npz`)
   -- artefak yang dipakai `backend/`.

> **Kernel wajib ada `torch`.** Prasyarat: `01_train.ipynb` sudah menghasilkan
> `checkpoints/best.pth`.

## 1. Setup

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

_p = Path.cwd()
while not ((_p / "src").is_dir() and (_p / "configs").is_dir()):
    if _p == _p.parent:
        raise RuntimeError("folder ml/ (berisi src/ & configs/) tidak ketemu")
    _p = _p.parent
os.chdir(_p)
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True
print("working dir:", os.getcwd())
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
from src.utils import set_seed, get_device, resolve_path
from src.dataset import WikiArtDataset
from src.transforms import build_eval_transforms
from src.model import build_model, load_checkpoint
from src.embedding import (extract_embeddings, save_embedding_index, nearest_neighbors,
                           evaluate_retrieval, chance_recall_at_k)
from torch.utils.data import DataLoader

CKPT = "checkpoints/v1_11class_backup/last.pth"

# checkpoint lama menyimpan config & label_to_idx SENDIRI (snapshot training v1) --
# sengaja TIDAK load configs/config.yaml yang sekarang (sudah berubah utk model baru).
_raw = torch.load(resolve_path(CKPT), map_location="cpu", weights_only=False)
cfg = _raw["config"]
cfg["train"]["num_workers"] = 0
set_seed(cfg["seed"])
device = get_device(cfg["device"])

label_to_idx = _raw["label_to_idx"]
idx_to_label = {v: k for k, v in label_to_idx.items()}
num_classes = len(label_to_idx)
MEAN = torch.tensor(cfg["image"]["mean"]).view(3, 1, 1)
STD  = torch.tensor(cfg["image"]["std"]).view(3, 1, 1)
def denorm(t): return (t * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

model = build_model(cfg, num_classes=num_classes).to(device)
ck = load_checkpoint(model, CKPT, map_location=str(device))
vm = ck["val_metrics"]
print(f"[MODEL LAMA v1] {model.arch} | emb_dim {model.embedding_dim} | checkpoint epoch {ck.get('epoch')} "
      f"| val_macro_f1={vm['macro_f1']:.4f}")
print("  (dari last.pth, bukan best.pth -- best.pth v1 sudah tertimpa training baru sebelum sempat dibackup)")

## 2. Ekstrak embedding: gallery (train+val) vs query (test)

Transform **deterministik** (`build_eval_transforms`) dipakai untuk ketiganya
-- embedding harus konsisten, tidak boleh diaugmentasi.

In [ ]:
eval_tf = build_eval_transforms(cfg)

def make_ds(split):
    return WikiArtDataset(
        csv_path=cfg["data"][f"{split}_csv"], image_dir=cfg["data"]["image_dir"],
        label_column=cfg["data"]["label_column"], label_to_idx=label_to_idx,
        transform=eval_tf, filename_column=cfg["data"]["filename_column"],
    )

emb_bs = cfg["embedding"]["batch_size"]
splits = {s: make_ds(s) for s in ("train", "val", "test")}
loaders_eval = {s: DataLoader(ds, batch_size=emb_bs, shuffle=False, num_workers=0)
               for s, ds in splits.items()}

embs, labs, fnames = {}, {}, {}
for s, ld in loaders_eval.items():
    e, l = extract_embeddings(model, ld, device,
                          l2_normalize=cfg["embedding"]["l2_normalize"], desc=f"embed:{s}")
    embs[s], labs[s] = e, l
    fnames[s] = splits[s].df[cfg["data"]["filename_column"]].tolist()
    print(f"{s:5s}: {e.shape[0]} gambar -> embedding {e.shape[1]}-d")

gallery_emb    = np.concatenate([embs["train"], embs["val"]], axis=0)
gallery_labels = np.concatenate([labs["train"], labs["val"]], axis=0)
gallery_fnames = fnames["train"] + fnames["val"]
query_emb, query_labels, query_fnames = embs["test"], labs["test"], fnames["test"]
print(f"\ngallery (train+val): {len(gallery_labels)} | query (test): {len(query_labels)}")

## 3. Recall@k / Precision@k / mAP@k — metrik sukses Visual Search

"Relevan" = gallery berlabel style sama dengan query. Dibandingkan ke
**chance baseline** (kalau retrieval dilakukan acak, seberapa sering "kena"
cuma karena kelas itu besar porsinya).

In [ ]:
KS = (1, 5, 10)
metrics = evaluate_retrieval(query_emb, query_labels, gallery_emb, gallery_labels, ks=KS)
chance = {k: chance_recall_at_k(gallery_labels, k) for k in KS}

print(f"{'k':>4} | {'Recall@k':>10} | {'chance':>8} | {'Precision@k':>12} | {'mAP@k':>8}")
for k in KS:
    print(f"{k:4d} | {metrics[f'recall@{k}']:10.3f} | {chance[k]:8.3f} | "
          f"{metrics[f'precision@{k}']:12.3f} | {metrics[f'map@{k}']:8.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(KS)); w = 0.35
ax.bar(x - w/2, [metrics[f"recall@{k}"] for k in KS], w, label="model")
ax.bar(x + w/2, [chance[k] for k in KS], w, label="chance (acak)")
ax.set_xticks(x, [f"Recall@{k}" for k in KS]); ax.set_ylim(0, 1); ax.legend()
ax.set_title("Retrieval: model vs baseline acak"); plt.tight_layout(); plt.show()

resolve_path("outputs/retrieval_metrics.json").write_text(
    json.dumps({"metrics": metrics, "chance": chance,
               "checkpoint": {"epoch": ck.get("epoch"), "monitor_value": ck.get("monitor_value")}},
              indent=2), encoding="utf-8")

## 4. Contoh nyata: 1 query -> top-5 hasil (yang dilihat user)

In [ ]:
rng = np.random.default_rng(cfg["seed"])
n_show = 4
q_idx = rng.choice(len(query_labels), size=n_show, replace=False)
idx, scores = nearest_neighbors(query_emb[q_idx], gallery_emb, top_k=5)

fig, axes = plt.subplots(n_show, 6, figsize=(18, 3.1 * n_show))
raw_dir = resolve_path(cfg["data"]["image_dir"])
from PIL import Image
for r, qi in enumerate(q_idx):
    qimg = Image.open(raw_dir / query_fnames[qi]).convert("RGB")
    axes[r, 0].imshow(qimg); axes[r, 0].set_title(f"QUERY\n{idx_to_label[query_labels[qi]]}", fontsize=9)
    axes[r, 0].axis("off")
    for c in range(5):
        gi = idx[r, c]
        gimg = Image.open(raw_dir / gallery_fnames[gi]).convert("RGB")
        ok = gallery_labels[gi] == query_labels[qi]
        axes[r, c+1].imshow(gimg)
        axes[r, c+1].set_title(f"#{c+1} sim={scores[r,c]:.2f}\n{idx_to_label[gallery_labels[gi]]}",
                               fontsize=8, color="green" if ok else "red")
        axes[r, c+1].axis("off")
fig.suptitle("Visual Search: query (kiri) -> top-5 hasil retrieval", fontsize=13)
plt.tight_layout(); plt.savefig(resolve_path("outputs/retrieval_examples.png"), dpi=100); plt.show()

## 5. Index embedding seluruh katalog (untuk `backend/`)

Meliputi SEMUA gambar (`embedding.source_csv` = metadata penuh), termasuk
kelas yang dibuang dari training (label -1 untuk itu) -- katalog Visual Search
tetap harus mencakup semua karya, bukan cuma 11 kelas yang dipakai training.

In [ ]:
REGEN_OUTPUT_PATH = "data/cache/v1_11class_backup/catalog_embeddings_regenerated.npz"

catalog_ds = WikiArtDataset(
    csv_path=cfg["embedding"]["source_csv"], image_dir=cfg["data"]["image_dir"],
    label_column=cfg["data"]["label_column"], label_to_idx=label_to_idx,
    transform=eval_tf, filename_column=cfg["data"]["filename_column"],
)
catalog_loader = DataLoader(catalog_ds, batch_size=cfg["embedding"]["batch_size"],
                           shuffle=False, num_workers=0)
cat_emb, cat_labels = extract_embeddings(model, catalog_loader, device,
                                         l2_normalize=cfg["embedding"]["l2_normalize"],
                                         desc="embed:katalog")
cat_fnames = catalog_ds.df[cfg["data"]["filename_column"]].tolist()
save_embedding_index(cat_emb, cat_fnames, cat_labels, REGEN_OUTPUT_PATH)
print(f"[REGENERATED] index katalog LAMA: {cat_emb.shape[0]} gambar x {cat_emb.shape[1]}-d -> "
      f"{REGEN_OUTPUT_PATH} (TIDAK menimpa live/backup asli)")
print(f"  ({int((cat_labels == -1).sum())} di antaranya di luar 11 kelas training, "
      f"tetap ter-embed)")

## 6. Uji instance retrieval — skenario "scan lukisan" (Google Lens-style)

Bedakan dua kebutuhan retrieval yang berbeda:

- **Similarity search** (Section 3–4 di atas): style sama -> Recall@1 0.81.
  Cocok untuk "jelajahi karya bergaya serupa".
- **Instance retrieval** (section ini): user foto **satu lukisan spesifik**
  (di internet / pameran / dunia nyata, mau beli) -> apakah sistem menemukan
  **lukisan itu PERSIS** di katalog kita (siapa penjualnya, pelukisnya,
  style-nya), bukan cuma yang gayanya kebetulan mirip? Ini use case nyata
  "scan untuk beli" yang jadi tujuan Visual Search.

Kita tidak punya foto asli "orang motret lukisan berkali-kali", jadi
disimulasikan: ambil gambar katalog, **distorsi** (rotasi, perspektif miring,
pencahayaan beda, blur ringan -- `src.transforms.build_capture_simulation_transform`,
meniru kondisi kamera HP), lalu cek apakah retrieval menemukan **file ASLI-nya**
(dicocokkan by identitas gambar, bukan cuma label style) sebagai hasil teratas.

In [ ]:
from src.transforms import build_capture_simulation_transform
from PIL import Image
from tqdm import tqdm

capture_tf = build_capture_simulation_transform(cfg)
raw_dir = resolve_path(cfg["data"]["image_dir"])

N_QUERY = 150
BATCH = 32
rng2 = np.random.default_rng(cfg["seed"])
query_idx = rng2.choice(len(cat_fnames), size=N_QUERY, replace=False)

sim_embs = []
with torch.no_grad():
    for start in tqdm(range(0, N_QUERY, BATCH), desc="simulasi foto HP"):
        batch_idx = query_idx[start:start + BATCH]
        imgs = [capture_tf(Image.open(raw_dir / cat_fnames[qi]).convert("RGB")) for qi in batch_idx]
        x = torch.stack(imgs).to(device)
        f = model.forward_features(x)
        if cfg["embedding"]["l2_normalize"]:
            f = torch.nn.functional.normalize(f, dim=1)
        sim_embs.append(f.cpu().numpy())
sim_embs = np.concatenate(sim_embs, axis=0)

# cari di SELURUH katalog (termasuk gambar aslinya sendiri -> itu yg diharapkan ketemu)
idxs, scores = nearest_neighbors(sim_embs, cat_emb, top_k=10)

hit1  = idxs[:, 0] == query_idx
hit5  = (idxs[:, :5]  == query_idx[:, None]).any(axis=1)
hit10 = (idxs[:, :10] == query_idx[:, None]).any(axis=1)
print(f"Instance retrieval (n={N_QUERY} simulasi 'foto HP'):")
print(f"  top-1 : {hit1.mean():.3f}")
print(f"  top-5 : {hit5.mean():.3f}")
print(f"  top-10: {hit10.mean():.3f}")

resolve_path("outputs/instance_retrieval_metrics.json").write_text(
    json.dumps({"n_query": N_QUERY, "top1": float(hit1.mean()),
               "top5": float(hit5.mean()), "top10": float(hit10.mean())},
              indent=2), encoding="utf-8")

In [ ]:
show_n = 3
fig, axes = plt.subplots(show_n, 4, figsize=(14, 3.2 * show_n))
for r in range(show_n):
    qi = query_idx[r]
    orig = Image.open(raw_dir / cat_fnames[qi]).convert("RGB")
    photo_tensor = capture_tf(Image.open(raw_dir / cat_fnames[qi]).convert("RGB"))
    axes[r, 0].imshow(orig); axes[r, 0].set_title("asli di katalog", fontsize=9); axes[r, 0].axis("off")
    axes[r, 1].imshow(denorm(photo_tensor)); axes[r, 1].set_title("simulasi foto HP\n(query)", fontsize=9)
    axes[r, 1].axis("off")
    for c in range(2):
        gi = idxs[r, c]
        gimg = Image.open(raw_dir / cat_fnames[gi]).convert("RGB")
        ok = gi == qi
        axes[r, 2 + c].imshow(gimg)
        axes[r, 2 + c].set_title(f"hasil #{c+1} sim={scores[r,c]:.2f}\n{'COCOK' if ok else 'beda'}",
                                 fontsize=9, color="green" if ok else "red")
        axes[r, 2 + c].axis("off")
fig.suptitle('Simulasi "scan lukisan" -> retrieval di katalog penuh', fontsize=13)
plt.tight_layout(); plt.savefig(resolve_path("outputs/instance_retrieval_examples.png"), dpi=100); plt.show()

## 7. Export ke TorchScript / ONNX (untuk `backend/`)

Tahap terakhir pipeline `ml/` (`preprocess -> train -> evaluate -> embedding
-> export`). `backend/` tidak menjalankan notebook Python -- dia butuh model
dalam format portable yang bisa di-load FastAPI. Output `forward()` di sini
= **embedding ternormalisasi L2** langsung (bukan logits), jadi `backend/`
tinggal panggil model lalu cari tetangga terdekat di katalog.

In [ ]:
from src.export import EmbeddingWrapper, export_torchscript, export_onnx

ex = cfg["export"]
export_model = EmbeddingWrapper(model, l2_normalize=cfg["embedding"]["l2_normalize"]).eval()
example = torch.randn(1, 3, cfg["image"]["size"], cfg["image"]["size"])

diff_ts = export_torchscript(export_model, example, ex["torchscript_path"])
print(f"TorchScript -> {ex['torchscript_path']}  (selisih vs PyTorch: {diff_ts:.2e})")

diff_onnx = export_onnx(export_model, example, ex["onnx_path"], ex["onnx_opset"])
if diff_onnx is None:
    print(f"ONNX -> {ex['onnx_path']}  (onnxruntime tidak ada, verifikasi dilewati)")
else:
    print(f"ONNX -> {ex['onnx_path']}  (selisih vs PyTorch: {diff_onnx:.2e})")

ok = diff_ts < 1e-4 and (diff_onnx is None or diff_onnx < 1e-4)
print("VERIFIKASI OK -- siap dipakai backend/" if ok else "[!] selisih > 1e-4, cek ulang")

## 8. Catatan interpretasi (untuk laporan)

- **Recall@k jauh di atas chance** = embedding benar-benar menangkap kemiripan
  visual/gaya, bukan kebetulan. Ini metrik yang dilaporkan sebagai "kinerja
  Visual Search", BUKAN akurasi klasifikasi style di `01_train.ipynb`.
- Query yang stylenya salah diklasifikasi (lihat confusion matrix `01_train`)
  **belum tentu retrieval-nya buruk** — dua lukisan bisa mirip secara visual
  walau beda label style (mis. Post-Impressionism vs Impressionism), dan itu
  hasil yang masuk akal untuk "cari karya mirip", bukan kegagalan.
- `data/cache/catalog_embeddings.npz` = artefak yang dikonsumsi `backend/`
  (endpoint `/api/visual-search`): terima foto -> `forward_features` ->
  `nearest_neighbors` ke index ini -> kembalikan karya termirip.
- Keterbatasan sama seperti `01_train.ipynb`: subset shard 0-4/72, tidak
  representatif, label style sebagai proksi "mirip" itu pendekatan, bukan
  ground truth kemiripan visual sebenarnya (yang idealnya dari studi user).